In [4]:
import os
import io
import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
import matplotlib.pyplot as plt
from google.colab import drive

In [5]:
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
root_path = '/content/drive/MyDrive/notebooks'

if not os.path.exists(root_path):
    os.makedirs(root_path)

os.chdir(root_path)
print(f"Current work folder: {os.getcwd()}")

Current work folder: /content/drive/MyDrive/notebooks


In [7]:
data_dir = os.path.join(root_path, 'crawler_data')

files = [
    'companies.csv',
    'companies_embeddings.csv',
    'jobs.csv',
    'jobs_embeddings.csv',
    'skills.csv',
    'skills_jobs.csv'
]

dataframes = {}

for file in files:
    file_path = os.path.join(data_dir, file)
    if os.path.exists(file_path):
        df_name = file.replace('.csv', '')
        dataframes[df_name] = pd.read_csv(file_path)
        print(f"Loaded {file} successfully.")
    else:
        print(f"Warning: {file} not found in {data_dir}")

if 'companies' in dataframes:
  companies_df = dataframes['companies']
if 'companies_embeddings' in dataframes:
  companies_embeddings_df = dataframes['companies_embeddings']
if 'jobs' in dataframes:
  jobs_df = dataframes['jobs']
if 'jobs_embeddings' in dataframes:
  jobs_embeddings_df = dataframes['jobs_embeddings']
if 'skills' in dataframes:
  skills_df = dataframes['skills']
if 'skills_jobs' in dataframes:
  skills_jobs_df = dataframes['skills_jobs']

Loaded companies.csv successfully.
Loaded companies_embeddings.csv successfully.
Loaded jobs.csv successfully.
Loaded jobs_embeddings.csv successfully.
Loaded skills.csv successfully.
Loaded skills_jobs.csv successfully.


In [8]:
df = [
    'companies_df',
    'companies_embeddings_df',
    'jobs_df',
    'jobs_embeddings_df',
    'skills_df',
    'skills_jobs_df'
]

for i in df:
  print(f"Info for {i}:")
  exec(f"{i}.info()")
  print("\n")


Info for companies_df:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 795 entries, 0 to 794
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id               795 non-null    int64 
 1   created_date     795 non-null    object
 2   updated_date     795 non-null    object
 3   name             795 non-null    object
 4   industry         791 non-null    object
 5   size             795 non-null    object
 6   location         795 non-null    object
 7   description      725 non-null    object
 8   website          390 non-null    object
 9   slogan           477 non-null    object
 10  company_type     795 non-null    object
 11  country          795 non-null    object
 12  addresses        795 non-null    object
 13  working_days     398 non-null    object
 14  overtime_policy  398 non-null    object
 15  vector_context   795 non-null    object
 16  benefits         795 non-null    object
dtypes: int64(1),

In [32]:
class DatasetQualityEvaluator:
    def __init__(self, companies_df, companies_embeddings_df, jobs_df, jobs_embeddings_df, skills_df, skills_jobs_df):
        self.companies_df = companies_df
        self.companies_embeddings_df = companies_embeddings_df
        self.jobs_df = jobs_df
        self.jobs_embeddings_df = jobs_embeddings_df
        self.skills_df = skills_df
        self.skills_jobs_df = skills_jobs_df
        self.metrics = {}

    def evaluate_completeness(self):
        def get_valid_mask(series):
            if series.dtype in ['float64', 'int64']:
                return series.notna()
            return ~(series.isna() | series.astype(str).str.strip().str.lower().isin(['', 'n/a', 'na', 'null', 'none', 'nan']))

        job_miss_cols = ['title', 'job_description', 'responsibilities', 'required_qualifications', 'nice_to_have', 'min_salary', 'max_salary', 'domains', 'source', 'url']
        job_valid_ratios = [get_valid_mask(self.jobs_df[col]).mean() for col in job_miss_cols]
        comp_miss_cols = ['name', 'industry', 'size', 'location', 'description', 'website', 'slogan', 'company_type', 'country', 'addresses', 'working_days', 'overtime_policy', 'benefits']
        comp_valid_ratios = [get_valid_mask(self.companies_df[col]).mean() for col in comp_miss_cols]
        missing_completeness = np.mean(job_valid_ratios + comp_valid_ratios) * 100 if (job_valid_ratios + comp_valid_ratios) else 0

        job_desc_len = self.jobs_df['job_description'].fillna("").astype(str).str.split().str.len()
        comp_desc_len = self.companies_df['description'].fillna("").astype(str).str.split().str.len()
        content_completeness = np.mean([(job_desc_len >= 50).mean(), (comp_desc_len >= 30).mean()]) * 100

        valid_job_companies = self.jobs_df['company_id'].isin(self.companies_df['id']).mean()
        valid_sj_jobs = self.skills_jobs_df['job_id'].isin(self.jobs_df['id']).mean() if not self.skills_jobs_df.empty else 0
        valid_sj_skills = self.skills_jobs_df['skill_id'].isin(self.skills_df['id']).mean() if not self.skills_jobs_df.empty else 0
        relationship_completeness = np.mean([valid_job_companies, valid_sj_jobs, valid_sj_skills]) * 100

        job_emb_complete = self.jobs_df['id'].isin(self.jobs_embeddings_df['job_id']).mean()
        comp_emb_complete = self.companies_df['id'].isin(self.companies_embeddings_df['company_id']).mean()
        embedding_completeness = np.mean([job_emb_complete, comp_emb_complete]) * 100

        jobs_with_skills = self.jobs_df['id'].isin(self.skills_jobs_df['job_id'])
        jobs_retrieval = (get_valid_mask(self.jobs_df['title']) &
                          get_valid_mask(self.jobs_df['job_description']) &
                          get_valid_mask(self.jobs_df['vector_context']) &
                          jobs_with_skills).mean()
        comp_retrieval = (get_valid_mask(self.companies_df['name']) &
                          get_valid_mask(self.companies_df['description']) &
                          get_valid_mask(self.companies_df['vector_context'])).mean()
        retrieval_completeness = np.mean([jobs_retrieval, comp_retrieval]) * 100

        job_meta_cols = ['url', 'working_model', 'min_salary', 'max_salary', 'source']
        job_meta_valid = [get_valid_mask(self.jobs_df[col]).mean() for col in job_meta_cols]
        comp_meta_cols = ['website', 'location', 'size', 'industry']
        comp_meta_valid = [get_valid_mask(self.companies_df[col]).mean() for col in comp_meta_cols]
        metadata_completeness = np.mean(job_meta_valid + comp_meta_valid) * 100 if (job_meta_valid + comp_meta_valid) else 0

        self.metrics['completeness_missing'] = missing_completeness
        self.metrics['completeness_content'] = content_completeness
        self.metrics['completeness_relationship'] = relationship_completeness
        self.metrics['completeness_embedding'] = embedding_completeness
        self.metrics['completeness_retrieval'] = retrieval_completeness
        self.metrics['completeness_metadata'] = metadata_completeness

        self.metrics['completeness_score'] = np.mean([
            missing_completeness,
            content_completeness,
            relationship_completeness,
            embedding_completeness,
            retrieval_completeness,
            metadata_completeness
        ])

        return self.metrics['completeness_score']

    def evaluate_uniqueness(self):
        job_id_unique = (len(self.jobs_df['id'].unique()) / len(self.jobs_df)) * 100 if len(self.jobs_df) > 0 else 0
        comp_id_unique = (len(self.companies_df['id'].unique()) / len(self.companies_df)) * 100 if len(self.companies_df) > 0 else 0
        job_content_unique = (len(self.jobs_df['content_hash'].unique()) / len(self.jobs_df)) * 100 if len(self.jobs_df) > 0 else 0

        self.metrics['uniqueness_score'] = (job_id_unique + comp_id_unique + job_content_unique) / 3
        return self.metrics['uniqueness_score']

    def evaluate_integrity(self):
        jobs_to_comp = self.jobs_df['company_id'].isin(self.companies_df['id']).mean() * 100 if len(self.jobs_df) > 0 else 0
        comp_to_jobs = self.companies_df['id'].isin(self.jobs_df['company_id']).mean() * 100 if len(self.companies_df) > 0 else 0

        j_emb_to_jobs = self.jobs_embeddings_df['job_id'].isin(self.jobs_df['id']).mean() * 100 if len(self.jobs_embeddings_df) > 0 else 0
        jobs_to_j_emb = self.jobs_df['id'].isin(self.jobs_embeddings_df['job_id']).mean() * 100 if len(self.jobs_df) > 0 else 0

        c_emb_to_comp = self.companies_embeddings_df['company_id'].isin(self.companies_df['id']).mean() * 100 if len(self.companies_embeddings_df) > 0 else 0
        comp_to_c_emb = self.companies_df['id'].isin(self.companies_embeddings_df['company_id']).mean() * 100 if len(self.companies_df) > 0 else 0

        if not self.skills_jobs_df.empty:
            sj_to_jobs = self.skills_jobs_df["job_id"].isin(self.jobs_df["id"]).mean() * 100
            sj_to_skills = self.skills_jobs_df["skill_id"].isin(self.skills_df["id"]).mean() * 100
            jobs_to_sj = self.jobs_df["id"].isin(self.skills_jobs_df["job_id"]).mean() * 100 if len(self.jobs_df) > 0 else 0
            skills_to_sj = self.skills_df["id"].isin(self.skills_jobs_df["skill_id"]).mean() * 100 if len(self.skills_df) > 0 else 0
        else:
            sj_to_jobs = sj_to_skills = jobs_to_sj = skills_to_sj = 0

        self.metrics['integrity_jobs_companies'] = (jobs_to_comp + comp_to_jobs) / 2
        self.metrics['integrity_jobs_embeddings'] = (j_emb_to_jobs + jobs_to_j_emb) / 2
        self.metrics['integrity_comp_embeddings'] = (c_emb_to_comp + comp_to_c_emb) / 2
        self.metrics['integrity_skills_jobs'] = (sj_to_jobs + sj_to_skills + jobs_to_sj + skills_to_sj) / 4

        self.metrics['integrity_score'] = (
            self.metrics['integrity_jobs_companies'] +
            self.metrics['integrity_jobs_embeddings'] +
            self.metrics['integrity_comp_embeddings'] +
            self.metrics['integrity_skills_jobs']
        ) / 4

        return self.metrics['integrity_score']

    def evaluate_rag_readiness(self):
        def is_valid_field(series):
            if series.dtype in ['float64', 'int64']:
                return series.notna()
            return ~(series.isna() | series.astype(str).str.strip().str.lower().isin(['', 'n/a', 'na', 'null', 'none', 'nan']))

        job_words = self.jobs_df['job_description'].fillna("").astype(str).str.split()
        comp_words = self.companies_df['description'].fillna("").astype(str).str.split()

        job_suff = (job_words.str.len() >= 100).mean() * 100 if len(job_words) > 0 else 0
        comp_suff = (comp_words.str.len() >= 50).mean() * 100 if len(comp_words) > 0 else 0
        score_sufficiency = (job_suff + comp_suff) / 2

        job_unique = job_words.apply(lambda x: len(set(x)))
        comp_unique = comp_words.apply(lambda x: len(set(x)))
        job_rich = (job_unique >= 50).mean() * 100 if len(job_unique) > 0 else 0
        comp_rich = (comp_unique >= 30).mean() * 100 if len(comp_unique) > 0 else 0
        score_richness = (job_rich + comp_rich) / 2

        job_struct = (is_valid_field(self.jobs_df['responsibilities']) &
                      is_valid_field(self.jobs_df['required_qualifications']) &
                      (is_valid_field(self.jobs_df['min_salary']) | is_valid_field(self.jobs_df['max_salary']))).mean() * 100 if len(self.jobs_df) > 0 else 0
        comp_struct = (is_valid_field(self.companies_df['description']) &
                       is_valid_field(self.companies_df['benefits']) &
                       is_valid_field(self.companies_df['location'])).mean() * 100 if len(self.companies_df) > 0 else 0
        score_structure = (job_struct + comp_struct) / 2

        jobs_with_skills = self.jobs_df['id'].isin(self.skills_jobs_df['job_id'])
        job_retrieval = (is_valid_field(self.jobs_df['vector_context']) &
                         is_valid_field(self.jobs_df['title']) &
                         is_valid_field(self.jobs_df['company_id']) &
                         jobs_with_skills).mean() * 100 if len(self.jobs_df) > 0 else 0
        comp_retrieval = (is_valid_field(self.companies_df['name']) &
                          is_valid_field(self.companies_df['description']) &
                          is_valid_field(self.companies_df['vector_context'])).mean() * 100 if len(self.companies_df) > 0 else 0
        score_retrieval = (job_retrieval + comp_retrieval) / 2

        def eval_embeddings(ref_df, emb_df, ref_id_col, emb_id_col):
            if len(ref_df) == 0:
                return 0
            merged = ref_df[[ref_id_col]].merge(emb_df, how='left', left_on=ref_id_col, right_on=emb_id_col)
            valid_emb = merged['embedding'].notna() & (merged['embedding'].astype(str).str.len() > 10)
            return valid_emb.mean() * 100

        job_emb_score = eval_embeddings(self.jobs_df, self.jobs_embeddings_df, 'id', 'job_id')
        comp_emb_score = eval_embeddings(self.companies_df, self.companies_embeddings_df, 'id', 'company_id')
        score_embedding = (job_emb_score + comp_emb_score) / 2

        self.metrics['rag_sufficiency'] = score_sufficiency
        self.metrics['rag_richness'] = score_richness
        self.metrics['rag_structure'] = score_structure
        self.metrics['rag_retrieval'] = score_retrieval
        self.metrics['rag_embedding'] = score_embedding

        self.metrics['rag_readiness_score'] = (
            (score_sufficiency * 0.20) +
            (score_richness * 0.25) +
            (score_structure * 0.20) +
            (score_retrieval * 0.20) +
            (score_embedding * 0.15)
        )

        return self.metrics['rag_readiness_score']

    def run_pipeline(self):
        self.evaluate_completeness()
        self.evaluate_uniqueness()
        self.evaluate_integrity()
        self.evaluate_rag_readiness()
        return self.metrics

    def visualize_results(self):
        total_companies = len(self.companies_df)
        total_jobs = len(self.jobs_df)
        total_skills = len(self.skills_df)
        total_embeddings = len(self.jobs_embeddings_df) + len(self.companies_embeddings_df)

        fig = go.Figure()

        fig.add_trace(go.Indicator(
            mode="number",
            value=total_companies,
            title={"text": "Total Companies"},
            domain={'x': [0, 0.22], 'y': [0.55, 1]}
        ))

        fig.add_trace(go.Indicator(
            mode="number",
            value=total_jobs,
            title={"text": "Total Jobs"},
            domain={'x': [0.26, 0.48], 'y': [0.55, 1]}
        ))

        fig.add_trace(go.Indicator(
            mode="number",
            value=total_skills,
            title={"text": "Total Skills"},
            domain={'x': [0.52, 0.74], 'y': [0.55, 1]}
        ))

        fig.add_trace(go.Indicator(
            mode="number",
            value=total_embeddings,
            title={"text": "Total Embeddings"},
            domain={'x': [0.78, 1], 'y': [0.55, 1]}
        ))

        fig.add_trace(go.Indicator(
            mode="number",
            value=self.metrics.get('completeness_score', 0),
            number={"suffix": "%", "valueformat": ".1f"},
            title={"text": "Completeness Score"},
            domain={'x': [0, 0.22], 'y': [0, 0.45]}
        ))

        fig.add_trace(go.Indicator(
            mode="number",
            value=self.metrics.get('uniqueness_score', 0),
            number={"suffix": "%", "valueformat": ".1f"},
            title={"text": "Uniqueness Score"},
            domain={'x': [0.26, 0.48], 'y': [0, 0.45]},
        ))

        fig.add_trace(go.Indicator(
            mode="number",
            value=self.metrics.get('integrity_score', 0),
            number={"suffix": "%", "valueformat": ".1f"},
            title={"text": "Integrity Score"},
            domain={'x': [0.52, 0.74], 'y': [0, 0.45]}
        ))

        fig.add_trace(go.Indicator(
            mode="number",
            value=self.metrics.get('rag_readiness_score', 0),
            number={"suffix": "%", "valueformat": ".1f"},
            title={"text": "RAG Quality Score"},
            domain={'x': [0.78, 1], 'y': [0, 0.45]}
        ))

        fig.update_layout(
            height=500,
            margin=dict(t=140),
            title={
                'text': "ADAPTIVE CRAWLER STATISTICS AND QUALITY RATING <br><sub>The dataset was collected on August 5, 2026.<sub>",
                'x': 0.5,
            },
            showlegend=False,
            paper_bgcolor="white",
            plot_bgcolor="white",
        )
        fig.show()

In [31]:
evaluator = DatasetQualityEvaluator(
    companies_df, companies_embeddings_df, jobs_df,
    jobs_embeddings_df, skills_df, skills_jobs_df
)

metrics = evaluator.run_pipeline()
evaluator.visualize_results()